# Piotroski F-Score — Validation Notebook

**Goal:** Compute F-Scores from v2 DB, compare against v1 output.  
**Pass criteria:** ≥95% of overlapping stocks within 1 point of v1.

v1 source: `~/alpha-signal/data/signals/piotroski.csv` (1,978 stocks)  
v2 source: `signals/piotroski.py` → reads quarterly_income + annual_balance_sheet + annual_cash_flow

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from signals.piotroski import _load_data, _compute_scores

## Step 1: Compute v2 scores

In [ ]:
stocks, qi, bs, cf = _load_data()
v2 = _compute_scores(stocks, qi, bs, cf)
print(f"v2 computed: {len(v2)} stocks, {v2['f_score'].notna().sum()} scored")
print(f"\nF-Score distribution:")
print(v2["f_score"].value_counts().sort_index())

## Step 2: Load v1 scores and merge

In [ ]:
v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/signals/piotroski.csv"))
print(f"v1: {len(v1)} stocks")

# Merge on sid
merged = v2.merge(
    v1[["sid", "piotroski_f_score", "f1_roa", "f2_ocf", "f3_delta_roa", 
        "f4_accruals", "f5_leverage", "f6_current_ratio", "f7_dilution",
        "f8_ebit_margin", "f9_asset_turnover", "data_quality"]],
    on="sid", how="inner", suffixes=("_v2", "_v1")
)
print(f"Overlapping stocks: {len(merged)}")
print(f"v2 has f_score: {merged['f_score'].notna().sum()}")
print(f"v1 has f_score: {merged['piotroski_f_score'].notna().sum()}")

## Step 3: Compare F-Scores — the key test

**Pass:** ≥95% of stocks where both have scores should be within ±1 point.

In [ ]:
# Filter to stocks where both v1 and v2 have scores
both = merged[merged["f_score"].notna() & merged["piotroski_f_score"].notna()].copy()
both["diff"] = both["f_score"] - both["piotroski_f_score"]
both["abs_diff"] = both["diff"].abs()

print(f"Stocks with both scores: {len(both)}")
print(f"\nDifference distribution (v2 - v1):")
print(both["diff"].value_counts().sort_index())
print(f"\n--- KEY METRIC ---")
within_1 = (both["abs_diff"] <= 1).sum()
pct = within_1 / len(both) * 100
print(f"Within ±1 point: {within_1}/{len(both)} = {pct:.1f}%")
print(f"{'PASS ✓' if pct >= 95 else 'FAIL ✗'}")

print(f"\nExact match: {(both['diff'] == 0).sum()}/{len(both)} = {(both['diff'] == 0).mean()*100:.1f}%")
print(f"Mean absolute diff: {both['abs_diff'].mean():.2f}")

## Step 4: Factor-by-factor comparison

Which individual factors differ most between v1 and v2?

In [ ]:
# Map v2 column names to v1 column names for comparison
factor_map = {
    "roa_positive":     "f1_roa",
    "cfo_positive":     "f2_ocf",
    "roa_improving":    "f3_delta_roa",
    "accruals_quality": "f4_accruals",
    "leverage_down":    "f5_leverage",
    "liquidity_up":     "f6_current_ratio",
    "no_dilution":      "f7_dilution",
    "gross_margin_up":  "f8_ebit_margin",
    "asset_turnover_up":"f9_asset_turnover",
}

print(f"{'Factor':<20} {'Both valid':>10} {'Match':>8} {'Match%':>8} {'v2 NaN':>7} {'v1 NaN':>7}")
print("-" * 62)

for v2_col, v1_col in factor_map.items():
    v2_vals = merged[v2_col].astype(float)
    v1_vals = merged[v1_col].astype(float)
    
    both_valid = v2_vals.notna() & v1_vals.notna()
    n_both = both_valid.sum()
    
    if n_both > 0:
        match = (v2_vals[both_valid] == v1_vals[both_valid]).sum()
        pct = match / n_both * 100
    else:
        match, pct = 0, 0.0
    
    v2_na = v2_vals.isna().sum()
    v1_na = v1_vals.isna().sum()
    
    print(f"{v2_col:<20} {n_both:>10} {match:>8} {pct:>7.1f}% {v2_na:>7} {v1_na:>7}")

## Step 5: Investigate mismatches (if any)

Look at the biggest outliers to understand *why* they differ.

In [ ]:
# Show stocks with biggest differences
outliers = both[both["abs_diff"] >= 2].sort_values("abs_diff", ascending=False)
if len(outliers) > 0:
    print(f"Stocks with |diff| >= 2: {len(outliers)}")
    display_cols = ["sid", "f_score", "piotroski_f_score", "diff", "data_quality"]
    print(outliers[display_cols].head(20).to_string(index=False))
else:
    print("No outliers with |diff| >= 2 — excellent!")

## Step 6: v2 distribution sanity check

Compare overall distributions — shape should be similar even if individual stocks differ.

In [ ]:
print("v1 F-Score stats:")
print(f"  Mean: {v1['piotroski_f_score'].mean():.2f}, Median: {v1['piotroski_f_score'].median():.1f}")
print(f"  Std:  {v1['piotroski_f_score'].std():.2f}")
print()
print("v2 F-Score stats:")
v2_scored = v2[v2["f_score"].notna()]["f_score"].astype(float)
print(f"  Mean: {v2_scored.mean():.2f}, Median: {v2_scored.median():.1f}")
print(f"  Std:  {v2_scored.std():.2f}")
print()
print("v2 coverage advantage:")
print(f"  v1: {len(v1)} stocks scored")
print(f"  v2: {v2['f_score'].notna().sum()} stocks scored")
print(f"  Net new: {v2['f_score'].notna().sum() - len(v1)}")

## Step 7: Save to DB (only if validation passed)

Run this cell only after confirming ≥95% match above.

In [ ]:
# from signals.piotroski import compute
# compute(dry_run=False)